# 48. 원질문을 보존하는 구조화 검색 보조

47번 선정 A(전체 복귀)를 고정합니다. 기존48개 개발 질문, 두 운영 프로필에서 A(원질문 깊이50), B(원질문50+구조화 보조50), A_budget(원질문100)을 비교합니다. 최종 후보20·카드3·근거5·64000bytes는 같습니다.

보조어는 45번에서 원문 좌표와 청크 연결이 확인된 실제 일치 fact의 필드 조각만 사용합니다. 불확실한 자료는 건너뛰며 추가 LLM은 없습니다. 출처 연결이 정확한 사실관계 전체를 보장하지는 않습니다. 전체 카드 범위와 원질문 재정렬은 유지합니다. 이 노트북에서 실행/재개/평가를 하고, 공통 인덱스 복사·예산·API 기록 책임만 기존47 보조 코드를 재사용합니다. HTML은 결과 확인 뒤 직접 작성합니다. 운영 수정은 없습니다.

In [1]:
from pathlib import Path
import importlib.util,sys,json,hashlib,time,os,copy
ROOT=Path.cwd()
if not (ROOT/'notebooks').exists():ROOT=ROOT.parent
sys.dont_write_bytecode=True
def load(name,filename):
 spec=importlib.util.spec_from_file_location(name,ROOT/'notebooks/scripts'/filename)
 module=importlib.util.module_from_spec(spec);sys.modules[name]=module;spec.loader.exec_module(module);return module
R=load('stage48_runtime','47_retrieval_runtime.py');P=R.P
PATH=P.F.STAGES[48];PATH.mkdir(exist_ok=True);P.F.initialize(48)
previous=P.F.STAGES[47]
assert P.F.read_json(previous/'completion_status.json')['status']=='complete'
assert not P.F.read_json(previous/'selected_policy.json')['partial_prioritization']
prior=P.DATA/'42_expanded_query_evaluation'
queries=P.F.read_json(prior/'questions.json')['queries'];catalog=P.F.read_json(prior/'catalog.json')['catalog']
annotations=P.F.read_json(prior/'dataset.json')['queries']
plans={r['query_id']:r for r in P.F.read_json(prior/'parser_results.json')['records']}
facts=[json.loads(s) for s in (P.DATA/'30_structured_benefit_db/facts.jsonl').read_text().splitlines()]
fact_by_id={f['fact_id']:f for f in facts}
mapping=P.F.read_json(P.DATA/'45_evidence_supplement_comparison/prepared_packages.json')['mapping']
profiles=['card_page_section_benefit','parent_child_bundle'];modes=['A','B','A_budget']
ALLOW_NETWORK=os.getenv('PREPROD48_ALLOW_NETWORK')=='1'
def write(name,value):P.F.write_json(PATH/name,value)
def frozen(name,value):
 path=PATH/name
 if path.exists():assert P.F.read_json(path)==value, '봉인 자료 변경: '+name
 else:write(name,value)
print('47번 선정 확인, 기존48개 개발 질문 사용')

47번 선정 확인, 기존48개 개발 질문 사용


In [2]:
# 원문 필드 조각을 질의 단서로만 쓰며 사용자 요구 조건으로 승격하지 않습니다.
FIELD_LABELS={'target':'혜택대상','action':'혜택방식','value':'혜택값','unit':'단위','condition':'적용조건',
 'cap':'한도','frequency':'횟수','period':'기간','exceptions':'제외조건'}
def guidance(routing,profile):
 if routing['reason']!='matched':return dict(text=None,used=[],skipped=[],reason='no_full_structured_match')
 accepted=[];skipped=[];blocks=[]
 selected=routing['selection']['matching_fact_ids']
 for card in sorted(selected):
  assert card in routing['scope']
  for fid in sorted(selected[card]):
   item=mapping.get(profile+':'+fid);fact=fact_by_id[fid]
   if item is None or item['status']!='verified':
    skipped.append(dict(fact_id=fid,reason='source_mapping_not_verified'));continue
   assert item['document_id']==fact['document_id']==card
   anchors=item['anchors']
   if not anchors or any(a['source_line'][a['char_start']:a['char_end']]!=a['fragment'] for a in anchors):
    skipped.append(dict(fact_id=fid,reason='strict_source_coordinate_mismatch'));continue
   parts=sorted({FIELD_LABELS[a['field']]+': '+a['fragment'] for a in anchors if a['field'] in FIELD_LABELS})
   block='[혜택 묶음]\n'+'\n'.join(parts)
   proposal=blocks if block in blocks else blocks+[block]
   text='혜택과 적용 조건 확인\n'+'\n\n'.join(proposal)
   if len(text.encode())>4096:
    skipped.append(dict(fact_id=fid,reason='whole_fact_exceeds_4096_byte_auxiliary_budget'));continue
   blocks=proposal;accepted.append(dict(fact_id=fid,card_key=card,anchors=anchors,query_block=block))
 if not accepted:return dict(text=None,used=[],skipped=skipped,reason='no_verified_fields')
 return dict(text='혜택과 적용 조건 확인\n'+'\n\n'.join(blocks),used=accepted,skipped=skipped,reason='verified_source_fields')
routes={q['query_id']:R.selected_route(q['question'],plans[q['query_id']]['plan'],facts,catalog,
 trusted_scope=P.explicit_scope(q['question'],catalog),parser_failed=plans[q['query_id']]['status']!='ok') for q in queries}
guides={q['query_id']+':'+pf:guidance(routes[q['query_id']],pf) for q in queries for pf in profiles}
frozen('guidance_inputs.json',dict(records=guides,questions_sha256=P.sha(queries),no_gold_labels=True,
 auxiliary_is_search_hint_not_user_constraint=True))
assert guidance(dict(reason='no_eligible_cards'),'card_page_section_benefit')['text'] is None
# 과거 verified 표기만 믿지 않고 좌표 오류를 실제 저장 반례로 검사합니다.
bad_id='fact_0bad15f24f2fb4df716a51e7';bad_card=fact_by_id[bad_id]['document_id']
bad_route=dict(reason='matched',scope=[bad_card],selection=dict(matching_fact_ids={bad_card:[bad_id]}))
assert guidance(bad_route,'card_page_section_benefit')['text'] is None
assert guidance(bad_route,'card_page_section_benefit')['skipped'][0]['reason']=='strict_source_coordinate_mismatch'
for identity,g in guides.items():
 if g['text']:
  assert len(g['text'].encode())<=4096 and all(x['card_key'] in routes[identity.split(':')[0]]['scope'] for x in g['used'])
  assert g==guidance(routes[identity.split(':')[0]],identity.split(':')[1])
  assert all(v['query_block'] in g['text'] for v in g['used'])
  assert g['text'].count('[혜택 묶음]')==len({v['query_block'] for v in g['used']})
print('원문좌표·범위·결정적 보조어·불확실성 제외 검사 통과; 보조 적용',sum(bool(g['text']) for g in guides.values()),'/96')

원문좌표·범위·결정적 보조어·불확실성 제외 검사 통과; 보조 적용 32 /96


In [3]:
import nbformat
notebook_path=ROOT/'notebooks/48_structured_retrieval_guidance.ipynb'
code_sha=P.sha([c.source for c in nbformat.read(notebook_path,4).cells if c.cell_type=='code'])
old_manifest=P.F.read_json(previous/'execution_manifest.json')
source_paths=[ROOT/p for p in old_manifest['source_sha256']]
data_paths=[ROOT/p for p in old_manifest['data_sha256']]+[
 previous/'selected_policy.json',previous/'completion_status.json',
 P.DATA/'45_evidence_supplement_comparison/prepared_packages.json',
 *[previous/folder/f"{q['query_id']}:{pf}:A.json" for folder in ['retrieval','answers'] for q in queries for pf in profiles]]
manifest=dict(stage=48,split='development',model='gpt-6-luna',reasoning='medium',answer_output=2400,
 question_count=48,profiles=profiles,modes=modes,logical_answer_conditions=288,
 questions_sha256=P.sha(queries),annotations_sha256=P.sha(annotations),notebook_code_sha256=code_sha,
 guidance_sha256=P.sha(guides),allowed_identities={'parser':[],'answer':[f"{q['query_id']}:{pf}:{m}" for q in queries for pf in profiles for m in modes]},
 allowed_embedding_text_sha256=sorted({P.sha(q['question']) for q in queries}|{P.sha(g['text']) for g in guides.values() if g['text']}),
 source_sha256={str(p.relative_to(ROOT)):P.sha(p.read_bytes()) for p in source_paths},
 data_sha256={str(p.relative_to(ROOT)):P.sha(p.read_bytes()) for p in data_paths},
 maximum_physical_requests=578,maximum_answer_requests=576,maximum_parser_requests=0,maximum_embedding_requests=2,
 maximum_validation_retries_per_answer=1,maximum_network_retries=0,global_cost_cap_usd='8',stage_cost_cap_usd='2',
 parameters=dict(A_component_depth=50,B_original_depth=50,B_auxiliary_depth=50,A_budget_depth=100,
 final_candidates=20,top_cards=3,evidence_per_card=5,payload_bytes=64000,vector_weight=.4,parent_bundle='budget5',
 merge='equal RRF k60 per modality, original query reranker',auxiliary_bytes=4096),
 selection_gate='B must improve over A and A_budget, no new core/important-condition/citation/error/card loss. Otherwise A.',
 historical_reuse='same original question and evidence hash, bound47 row source; equal current inputs may reuse one answer',
 operating_changes=False,new_document_embeddings=0)
assert manifest['maximum_answer_requests']==manifest['logical_answer_conditions']*(1+manifest['maximum_validation_retries_per_answer'])
assert manifest['maximum_physical_requests']==manifest['maximum_answer_requests']+manifest['maximum_embedding_requests']
frozen('execution_manifest.json',manifest);frozen('questions.json',dict(queries=queries,split='development'))
frozen('evaluation_annotations.json',dict(queries=annotations,never_sent_to_provider=True))
# 현재 실제 파일을 기준으로 과거 원본 보존 목록을 확장합니다.
if not (PATH/'protected_files.json').exists():
 protected={r['path']:P.F.file_record(ROOT/r['path']) for r in P.F.read_json(previous/'protected_files.json')['files']}
 for source in previous.rglob('*'):
  if source.is_file() and not any(t in source.parts for t in ['indexes','execution_cache','contract_tests']):protected[str(source.relative_to(ROOT))]=P.F.file_record(source)
 for source in [ROOT/'notebooks/47_partial_condition_recovery.ipynb',ROOT/'notebooks/scripts/47_retrieval_runtime.py',ROOT/'docs/report/47_partial_condition_recovery_report.html']:
  protected[str(source.relative_to(ROOT))]=P.F.file_record(source)
 write('protected_files.json',dict(files=sorted(protected.values(),key=lambda r:r['path'])))
# 요청 identity와 실제 입력이 다르면 네트워크 전에 거절합니다.
from pickcardu_rag.answering import _generated_answer_model
fake=R.BoundCache(48,'answer','outside',allow_network=False)
try:fake.reserve(dict(input=[],text_format=_generated_answer_model(1)))
except (FileNotFoundError,KeyError,RuntimeError):pass
else:raise AssertionError('유료 입력 결합 거절 누락')
print('48×2×3=288조건, 전체8달러/단계2달러 상한 봉인. 이 셀까지 API0회.')

48×2×3=288조건, 전체8달러/단계2달러 상한 봉인. 이 셀까지 API0회.


In [4]:
runtime=R.Runtime(48)
texts=[q['question'] for q in queries]+sorted({g['text'] for g in guides.values() if g['text']})
vectors=runtime.embeddings(texts,allow_network=ALLOW_NETWORK);vector_by_text=dict(zip(texts,vectors,strict=True))
for pf in profiles:runtime.profile(pf)
retrievals=[]
for q in queries:
 route=routes[q['query_id']]
 for pf in profiles:
  cached_A=None;g=guides[q['query_id']+':'+pf]
  for mode in modes:
   identity=f"{q['query_id']}:{pf}:{mode}";path=PATH/'retrieval'/(identity+'.json')
   binding=P.sha(dict(question=q['question'],route=route,guide=g,profile=pf,mode=mode,
    manifest=P.sha(manifest),index=runtime.index_identity[pf]))
   if path.exists():
    row=P.F.read_json(path);assert row['input_binding']==binding and row['identity']==identity
   else:
    start=time.perf_counter()
    if mode=='B' and not g['text']:result=copy.deepcopy(cached_A['result']);reuse='current_A_no_guidance'
    else:
     supplement=dict(question=g['text'],vector=vector_by_text[g['text']],scope=route['scope']) if mode=='B' else None
     result=runtime.search(q['question'],vector_by_text[q['question']],pf,route['scope'],
      supplement=supplement,depth=100 if mode=='A_budget' else 50,
      baseline_result=cached_A['result'] if mode=='B' else None);reuse=None
    row=dict(identity=identity,query_id=q['query_id'],profile=pf,mode=mode,status='ok',result=result,
     route=route,guidance_applied=mode=='B' and bool(g['text']),input_binding=binding,reuse=reuse,
     wall_seconds=time.perf_counter()-start,manifest_sha256=P.sha(manifest))
    write('retrieval/'+identity+'.json',row)
   if mode=='A':cached_A=row
   retrievals.append(row)
   write('progress.json',dict(phase='retrieval',completed=len(retrievals),expected=288,last_identity=identity,ledger=P.ledger()))
   print('retrieval',identity,flush=True)
assert len(retrievals)==288 and len({r['identity'] for r in retrievals})==288
write('retrieval_results.json',dict(records=retrievals,resources=runtime.resources()))
from pickcardu_rag.answering import serialize_answer_payload
by_q={q['query_id']:q for q in queries}
payloads={r['identity']:P.sha([dict(role='user',content=serialize_answer_payload(by_q[r['query_id']]['question'],r['result']['evidence']))]) for r in retrievals}
frozen('answer_payload_contract.json',dict(execution_manifest_sha256=P.sha(manifest),input_sha256_by_identity=payloads,
 schema_sha256_by_identity={r['identity']:P.sha(_generated_answer_model(len(r['result']['evidence'])).model_json_schema()) for r in retrievals},
 retrieval_results_sha256=P.sha(retrievals),gold_labels_in_payload=False))
print('검색288조건 완료. 답변 입력 봉인',P.ledger())

/home/sms/miniconda3/envs/skn25/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


retrieval E01:card_page_section_benefit:A


retrieval E01:card_page_section_benefit:B


retrieval E01:card_page_section_benefit:A_budget


retrieval E01:parent_child_bundle:A


...

retrieval E48:card_page_section_benefit:A_budget


retrieval E48:parent_child_bundle:A


retrieval E48:parent_child_bundle:B


retrieval E48:parent_child_bundle:A_budget


검색288조건 완료. 답변 입력 봉인 {'record_count': 158, 'known_priced_records': 158, 'unknown_cost_records': 0, 'duplicate_saved_responses': 0, 'known_cost_usd': '0.124203180', 'tokens': {'input_tokens': 853067, 'output_tokens': 117504, 'ordinary_input_tokens': 468, 'cached_input_tokens': 359277, 'cache_write_tokens': 493322, 'embedding_tokens': 7318}, 'billed_cost_usd': None}


In [5]:
from pickcardu_rag.answering import OpenAIService
answers=[];by_payload={}
# 검증 완료한47번 A의 원입력·출력 파일을 현재 manifest의 해시로 결속합니다.
for q in queries:
 for pf in profiles:
  old=P.F.read_json(previous/'answers'/f"{q['query_id']}:{pf}:A.json")
  assert old['status']=='ok';by_payload[old['input_sha256']]=old
answer_contract=P.F.read_json(PATH/'answer_payload_contract.json')
for r in retrievals:
 identity=r['identity'];q=by_q[r['query_id']];path=PATH/'answers'/(identity+'.json');payload=payloads[identity]
 reused=by_payload.get(payload)
 if path.exists():
  row=P.F.read_json(path);assert row['retrieval_sha256']==P.sha(r) and row['identity']==identity
 else:
  start=time.perf_counter();events=[];error=None
  if reused is not None:answer=copy.deepcopy(reused['answer']);usage={};reuse=reused['identity']
  elif not r['result']['evidence']:
   answer=dict(answer_status='insufficient_evidence',answer_text='검색된 근거가 없습니다.',recommendations=[],claims=[]);usage={};reuse=None
  else:
   api=R.BoundCache(48,'answer',identity,allow_network=ALLOW_NETWORK);reuse=None
   try:
    parsed,usage=OpenAIService(api_key=None,client=api,llm_model='gpt-6-luna',answer_payload_bytes=64000).answer(q['question'],r['result']['evidence'])
    answer=parsed.model_dump(mode='json')
   except Exception as exc:
    if not api.events:raise
    answer=None;usage={};error=f'{type(exc).__name__}: {exc}'
   events=api.events
  row=dict(identity=identity,query_id=q['query_id'],profile=r['profile'],mode=r['mode'],status='ok' if answer else 'failed',
   answer=answer,usage=usage,events=events,error=error,reuse=reuse,retrieval_sha256=P.sha(r),input_sha256=payload,
   wall_seconds=time.perf_counter()-start,manifest_sha256=P.sha(manifest),payload_contract_sha256=P.sha(answer_contract),
   request_keys=[e['key'] for e in events],reuse_source_sha256=P.sha(reused) if reused is not None else None)
  write('answers/'+identity+'.json',row)
 R.validate_generated_answer(row,q['question'],r['result']['evidence'],stage=48,identity=identity,manifest=manifest,
  contract=answer_contract,reused=reused)
 answers.append(row)
 if row['status']=='ok':by_payload[payload]=row
 write('progress.json',dict(phase='answer',completed=len(answers),expected=288,last_identity=identity,ledger=P.ledger()))
 print('answer',identity,row['status'],row['reuse'] or 'fresh',flush=True)
assert len(answers)==288
write('answer_results.json',dict(records=answers,resources=runtime.resources()))
print('답변 완료',P.ledger())

answer E01:card_page_section_benefit:A ok E01:card_page_section_benefit:A


answer E01:card_page_section_benefit:B ok fresh


answer E01:card_page_section_benefit:A_budget ok fresh


answer E01:parent_child_bundle:A ok E01:parent_child_bundle:A


answer E01:parent_child_bundle:B ok E01:parent_child_bundle:A


...

answer E48:card_page_section_benefit:A_budget ok fresh


answer E48:parent_child_bundle:A ok E48:parent_child_bundle:A


answer E48:parent_child_bundle:B ok E48:parent_child_bundle:A


answer E48:parent_child_bundle:A_budget ok fresh


답변 완료 {'record_count': 158, 'known_priced_records': 158, 'unknown_cost_records': 0, 'duplicate_saved_responses': 0, 'known_cost_usd': '0.124203180', 'tokens': {'input_tokens': 853067, 'output_tokens': 117504, 'ordinary_input_tokens': 468, 'cached_input_tokens': 359277, 'cache_write_tokens': 493322, 'embedding_tokens': 7318}, 'billed_cost_usd': None}


In [6]:
M=load('stage48_rubric','39_answer_rubric.py');labels={q['query_id']:{**q,'plan':q['gold_plan']} for q in annotations}
lookup={r['identity']:r for r in retrievals};rows=[]
for a in answers:
 r=lookup[a['identity']];case=labels[a['query_id']];gold=set(case['gold_cards']);cards=[c['card_key'] for c in r['result']['cards']]
 rank=next((i for i,k in enumerate(cards,1) if k in gold),None)
 scores=M.score(case,a['answer'],r['result']['evidence']) if a['status']=='ok' else dict(known_positive_complete=0 if gold else None,citation_ownership=0,explicit_scope_preserved=0)
 rows.append(dict(identity=a['identity'],query_id=a['query_id'],profile=a['profile'],mode=a['mode'],subgroup=case['subgroup'],
  scores=scores,gold_hit_at_3=int(rank is not None) if gold else None,gold_mrr=1/rank if rank else 0 if gold else None,
  guidance_available=bool(guides[a['query_id']+':'+a['profile']]['text']),fresh_answer=bool(a['events'])))
pairs=[]
for q in queries:
 for pf in profiles:
  base=lookup[f"{q['query_id']}:{pf}:A"]
  for mode in ['B','A_budget']:
   other=lookup[f"{q['query_id']}:{pf}:{mode}"]
   pairs.append(dict(query_id=q['query_id'],profile=pf,mode=mode,same_evidence=base['result']['evidence']==other['result']['evidence'],
    original_scope_unchanged=base['route']['scope']==other['route']['scope'],
    original_evidence_lost=sorted({v['chunk_id'] for v in base['result']['evidence']}-{v['chunk_id'] for v in other['result']['evidence']})))
write('evaluation_results.json',dict(records=rows,pairs=pairs,resources=runtime.resources(),manual_source_review_required=True))
if not (PATH/'initial_execution_resources.json').exists():write('initial_execution_resources.json',dict(resources=runtime.resources(),initial_only=True))
else:write('latest_replay_resources.json',dict(resources=runtime.resources(),offline=not ALLOW_NETWORK))
assert all(p['original_scope_unchanged'] for p in pairs)
print('A 대비 근거 변경',sum(not p['same_evidence'] for p in pairs),'/192 비교')
changed=[];allowed=[]
allowance=P.F.read_json(PATH/'authorized_metric_append.json') if (PATH/'authorized_metric_append.json').exists() else None
for record in P.F.read_json(PATH/'protected_files.json')['files']:
 source=ROOT/record['path']
 if not source.exists():changed.append(record['path']);continue
 if P.F.file_record(source)!=record:
  body=source.read_bytes()
  if allowance and record['path']==allowance['path'] and P.sha(body)==allowance['current_sha256'] and P.sha(body[:record['size']])==record['sha256']:allowed.append(record['path'])
  else:changed.append(record['path'])
write('preservation_verification.json',dict(changed=changed,allowed_document_appends=allowed,passed=not changed))
assert not changed,changed
print('원본 보존 통과, 원문 검토 후 선정')

A 대비 근거 변경 68 /192 비교


원본 보존 통과, 원문 검토 후 선정
